<a href="https://colab.research.google.com/github/vanesamrd/ML_ganjil/blob/main/Vanesa_JS15_P02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Praktikum 2 - Deployment ke Hugging Face (Docker + Flask)

Pada praktikum ini dilakukan siklus lengkap pengembangan Machine Learning (MLOps sederhana): men-deploy model klasifikasi gambar **Day vs Night** (HOG + Neural Network) ke server cloud **Hugging Face Spaces** menggunakan **Docker container** dan **Flask**.

> Notebook ini membuat seluruh file deployment secara otomatis dan memverifikasi bahwa aplikasi Flask benar-benar bisa memuat model serta memprediksi gambar. Proses `git push` ke Hugging Face dilakukan manual karena membutuhkan akun dan Access Token.


## Langkah 1: Persiapan Environment

Menyiapkan struktur direktori kerja `daynight-project` dan memastikan file model (`day_night_model.h5`) serta scaler (`scaler.pkl`) hasil training tersedia.

In [1]:
from pathlib import Path
import shutil

project_dir = Path("daynight-project")
project_dir.mkdir(exist_ok=True)

# Salin artefak hasil training dari Praktikum 1
for filename in ["day_night_model.h5", "scaler.pkl"]:
    src = Path(filename)
    dst = project_dir / filename
    if not src.exists():
        raise FileNotFoundError(f"File {filename} tidak ditemukan. Jalankan Praktikum 1 terlebih dahulu.")
    shutil.copy2(src, dst)
    print(f"Berhasil menyalin {filename} -> {dst}")

print("\nIsi folder proyek:")
for f in sorted(project_dir.iterdir()):
    print(f" - {f.name} ({f.stat().st_size / 1024 / 1024:.2f} MB)")

Berhasil menyalin day_night_model.h5 -> daynight-project/day_night_model.h5
Berhasil menyalin scaler.pkl -> daynight-project/scaler.pkl

Isi folder proyek:
 - day_night_model.h5 (50.80 MB)
 - scaler.pkl (0.79 MB)


## Langkah 2: Konfigurasi Server (app.py)

Aplikasi web sederhana menggunakan Flask untuk menangani request gambar dari pengguna. Kode memuat model dan melakukan preprocessing HOG yang sama persis dengan saat training.

In [2]:
%%writefile daynight-project/app.py
import os
import numpy as np
import cv2
import pickle
import tensorflow as tf
from flask import Flask, request, render_template_string
from skimage.feature import hog

app = Flask(__name__)

# Load Model & Scaler
MODEL_PATH = 'day_night_model.h5'
SCALER_PATH = 'scaler.pkl'

try:
    model = tf.keras.models.load_model(MODEL_PATH)
    with open(SCALER_PATH, 'rb') as f:
        scaler = pickle.load(f)
    print("\u2705 System Loaded Successfully")
except Exception as e:
    print(f"\u274c Error loading system: {e}")

def preprocess_image(image_bytes):
    # Decode gambar
    nparr = np.frombuffer(image_bytes, np.uint8)
    img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)

    # Preprocessing (Harus sama persis dengan Training)
    img = cv2.resize(img, (256, 256))
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    hog_feat = hog(gray, orientations=9, pixels_per_cell=(8, 8),
                   cells_per_block=(2, 2), block_norm='L2-Hys',
                   visualize=False, feature_vector=True)

    return scaler.transform(hog_feat.reshape(1, -1))

@app.route('/', methods=['GET'])
def home():
    return render_template_string('''
    <div style="text-align:center; padding:50px;">
        <h1>Day vs Night Classifier</h1>
        <form action="/predict" method="post" enctype="multipart/form-data">
            <input type="file" name="file" required><br><br>
            <button type="submit">Prediksi</button>
        </form>
    </div>
    ''')

@app.route('/predict', methods=['POST'])
def predict():
    try:
        file = request.files['file']
        data = preprocess_image(file.read())
        prediction = model.predict(data)[0][0]

        label = "Day (Siang)" if prediction > 0.5 else "Night (Malam)"
        return f"<h2 style='text-align:center'>Hasil: {label}</h2><center><a href='/'>Kembali</a></center>"
    except Exception as e:
        return f"Error: {e}"

if __name__ == '__main__':
    # Port 7860 wajib untuk Hugging Face Spaces
    app.run(host='0.0.0.0', port=7860)

Writing daynight-project/app.py


## Langkah 3: Konfigurasi Dependencies (requirements.txt)

Agar server cloud tahu library apa saja yang dibutuhkan, buat daftar dependencies.

In [3]:
%%writefile daynight-project/requirements.txt
flask
tensorflow-cpu
numpy
scikit-learn
scikit-image
opencv-python-headless
gunicorn

Writing daynight-project/requirements.txt


## Langkah 4: Konfigurasi Docker (Dockerfile)

Dockerfile berisi instruksi untuk membangun image sistem operasi yang akan menjalankan aplikasi.

In [4]:
%%writefile daynight-project/Dockerfile
# Base Image Python 3.9 Slim
FROM python:3.9-slim

# Install Library Sistem untuk OpenCV (Debian Bookworm/Trixie Compatible)
RUN apt-get update && apt-get install -y \
    libgl1 \
    libglib2.0-0 \
    && rm -rf /var/lib/apt/lists/*

# Setup User Non-Root (Standar Keamanan HF)
RUN useradd -m -u 1000 user
USER user
ENV PATH="/home/user/.local/bin:$PATH"

# Setup Direktori Kerja
WORKDIR /app

# Install Dependencies
COPY --chown=user ./requirements.txt requirements.txt
RUN pip install --no-cache-dir --upgrade -r requirements.txt

# Copy File Aplikasi
COPY --chown=user . /app

# Expose Port & Jalankan
EXPOSE 7860
CMD ["python", "app.py"]

Writing daynight-project/Dockerfile


## Langkah 5: Setup Hugging Face Space

1. Login ke akun [Hugging Face](https://huggingface.co/).
2. Klik foto profil pojok kanan atas, lalu pilih **New Space**.
3. Isi konfigurasi:
   - **Space Name:** `daynight-classifier-namaanda` (ganti `namaanda` dengan nama/NIM Anda)
   - **License:** Apache 2.0 / MIT
   - **Space SDK:** **Docker**
   - **Template:** kosongkan
   - **Visibility:** Public
4. Klik **Create Space**.
5. Buat Access Token: **Settings > Access Tokens > Create new token**, pilih permission **Write**, lalu salin token dan simpan sementara.


## Langkah 6: Upload ke Cloud (Hugging Face)

Jalankan perintah berikut di terminal (Git Bash/CMD) dari direktori kerja, ganti `USERNAME_ANDA` dan `NAMA_SPACE` sesuai milik Anda.

**1. Clone repository kosong dari Hugging Face**

```bash
git clone https://huggingface.co/spaces/USERNAME_ANDA/NAMA_SPACE
```

**2. Salin file proyek ke folder hasil clone**

Salin `app.py`, `Dockerfile`, `requirements.txt`, `day_night_model.h5`, dan `scaler.pkl` dari folder `daynight-project` ke folder hasil clone.

**3. Upload file ke Hugging Face**

```bash
cd NAMA_SPACE
git add .
git commit -m "Deploy aplikasi ke HF"
git push
```

**4. Autentikasi**

Saat `git push` meminta kredensial:

- **Username:** username Hugging Face Anda.
- **Password:** tempelkan **Access Token** (bukan password akun) yang dibuat pada Langkah 5.


## Langkah 7: Pengujian dan Hasil Akhir

Sebelum di-push, aplikasi diuji terlebih dahulu secara lokal menggunakan **Flask test client** agar tidak perlu menjalankan server secara manual. Pengujian memakai satu gambar siang dan satu gambar malam dari dataset.

In [5]:
from pathlib import Path
import os, sys, io

repo_dir = Path.cwd()
project_dir = Path("daynight-project").resolve()
images_dir = Path("images").resolve()

print("Isi folder deployment:")
for f in sorted(project_dir.iterdir()):
    size = f"{f.stat().st_size / 1024 / 1024:.2f} MB" if f.is_file() else ""
    print(f" - {f.name} {size}")

# Import aplikasi Flask dari folder proyek
sys.path.insert(0, str(project_dir))
os.chdir(project_dir)
if "app" in sys.modules:
    del sys.modules["app"]
import app as webapp
os.chdir(repo_dir)

client = webapp.app.test_client()
print("\nGET / ->", client.get("/").status_code)

print("\nHasil prediksi:")
for label in ["day", "night"]:
    sample = next((images_dir / "test" / label).glob("*.jpg"))
    with open(sample, "rb") as f:
        buf = io.BytesIO(f.read())
    resp = client.post("/predict", data={"file": (buf, sample.name)},
                       content_type="multipart/form-data")
    html = resp.data.decode()
    hasil = html.split("Hasil:")[1].split("</h2>")[0].strip() if "Hasil:" in html else html
    print(f" - {label:5s} | {sample.name} -> {hasil}")

Isi folder deployment:
 - Dockerfile 0.00 MB
 - app.py 0.00 MB
 - day_night_model.h5 50.80 MB
 - requirements.txt 0.00 MB
 - scaler.pkl 0.79 MB


/Users/vanesshh/Praktikum_JST/venv_tf/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


✅ System Loaded Successfully

GET / -> 200

Hasil prediksi:
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step


 - day   | 20151102_122322.jpg -> Day (Siang)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step


 - night | 20151102_051130.jpg -> Night (Malam)


### Hasil Akhir

Setelah `git push` selesai 100%, buka halaman Space dan klik tab **App**:

1. Status berubah menjadi **Building** (biru), tunggu 3-5 menit.
2. Jika sudah **Running** (hijau), unggah gambar uji (siang/malam) dan pastikan prediksi berjalan lancar.
